# NLP Lab Programs — 7 to 32

Programs 1–6 are excluded because they were completed earlier. This notebook contains each remaining program in a separate section.

**How to use:** Run the setup cell first, then run each program section individually. Some programs download pretrained models and may take several minutes.


## Setup
Run this cell once in a fresh Google Colab runtime.


In [ ]:
%pip install -q nltk spacy scikit-learn pandas gensim textblob vaderSentiment fastcoref transformers datasets accelerate sentencepiece tensorflow
!python -m spacy download en_core_web_sm


---

## Program 7: Coreference Resolution

**Course outcome:** CO2


In [ ]:
# Program 7: Coreference Resolution using FastCoref (neuralcoref alternative)
# Colab: !pip install -q spacy fastcoref
# Colab: !python -m spacy download en_core_web_sm
import spacy
from fastcoref import FCoref
text = "Riya bought a laptop yesterday. She said it was fast, and she uses the laptop for classes."
nlp = spacy.load("en_core_web_sm")
model = FCoref(device="cpu")
prediction = model.predict(texts=[text])[0]
print("Text:", text)
print("Coreference clusters:")
for i, cluster in enumerate(prediction.get_clusters(as_strings=True), 1):
    print(f"Cluster {i}: {cluster}")


---

## Program 8: Bag Of Words

**Course outcome:** CO2


In [ ]:
# Program 8: Bag-of-Words vectorization and representation
from sklearn.feature_extraction.text import CountVectorizer
docs = ["NLP helps computers understand language", "Computers process language using NLP", "Language processing is useful"]
v = CountVectorizer()
x = v.fit_transform(docs)
print("Vocabulary:", v.get_feature_names_out())
print("BoW matrix (documents x words):")
print(x.toarray())


---

## Program 9: Tfidf Vs Bow

**Course outcome:** CO2


In [ ]:
# Program 9: TF-IDF implementation and comparison with BoW
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
docs = ["machine learning processes text", "text mining uses machine learning", "deep learning processes images"]
bow = CountVectorizer()
b = bow.fit_transform(docs)
tf = TfidfVectorizer()
t = tf.fit_transform(docs)
print("BoW:")
print(pd.DataFrame(b.toarray(), columns=bow.get_feature_names_out()))
print("\nTF-IDF:")
print(pd.DataFrame(t.toarray().round(3), columns=tf.get_feature_names_out()))
print("\nBoW uses raw counts; TF-IDF downweights terms common across documents.")


---

## Program 10: Ngram Model

**Course outcome:** CO2


In [ ]:
# Program 10: Generate unigram, bigram, and trigram models
from sklearn.feature_extraction.text import CountVectorizer
corpus = ["natural language processing is useful", "natural language models learn patterns", "language processing uses statistical models"]
for n, name in [(1, "Unigrams"), (2, "Bigrams"), (3, "Trigrams")]:
    v = CountVectorizer(ngram_range=(n, n))
    x = v.fit_transform(corpus)
    print("\n" + name)
    for term, count in zip(v.get_feature_names_out(), x.toarray().sum(axis=0)):
        print(f"{term}: {count}")


---

## Program 11: Cosine Similarity

**Course outcome:** CO2


In [ ]:
# Program 11: Cosine similarity between text documents
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
docs = ["The cat sits on the mat", "A cat is sitting on a mat", "The stock market changed today"]
x = TfidfVectorizer().fit_transform(docs)
scores = cosine_similarity(x)
print(scores.round(3))
print("Similarity document 1 vs 2:", round(scores[0, 1], 3))
print("Similarity document 1 vs 3:", round(scores[0, 2], 3))


---

## Program 12: Word2Vec Gensim

**Course outcome:** CO2


In [ ]:
# Program 12: Word2Vec embeddings using gensim on a custom corpus
# Colab: !pip install -q gensim nltk
from gensim.models import Word2Vec
from nltk.tokenize import wordpunct_tokenize
corpus = ["nlp models learn words from context", "word embeddings represent words as vectors", "context helps models learn word meaning", "nlp uses embeddings for language tasks", "models learn patterns from text"]
sentences = [wordpunct_tokenize(s.lower()) for s in corpus]
model = Word2Vec(sentences, vector_size=50, window=3, min_count=1, workers=1, sg=1, seed=42, epochs=100)
print("Vector for models:", model.wv["models"])
print("Similar words:", model.wv.most_similar("models", topn=3))


---

## Program 13: Glove Embeddings

**Course outcome:** CO2


In [ ]:
# Program 13: Load GloVe embeddings and inspect vectors
# Colab: !pip install -q gensim
import gensim.downloader as api
print("Downloading/loading GloVe model; first run may take a few minutes.")
glove = api.load("glove-wiki-gigaword-50")
for word in ["computer", "language", "king"]:
    if word in glove:
        print(word, "shape:", glove[word].shape, "first 10 values:", glove[word][:10])
print("Words similar to computer:", glove.most_similar("computer", topn=5))


---

## Program 14: Word Movers Distance

**Course outcome:** CO2


In [ ]:
# Program 14: Text similarity using Word Mover's Distance (WMD)
# Colab: !pip install -q gensim nltk
import re, nltk, gensim.downloader as api
nltk.download("stopwords")
from nltk.corpus import stopwords
print("Loading GloVe vectors; first run may take a few minutes.")
vectors = api.load("glove-wiki-gigaword-50")
stop = set(stopwords.words("english"))
def tokens(s):
    return [w for w in re.findall(r"[a-z]+", s.lower()) if w not in stop and w in vectors]
a = tokens("the cat sits on the mat")
b = tokens("a cat is sitting on a rug")
c = tokens("the government announced new economic policy")
print("Doc1 tokens:", a, "\nDoc2 tokens:", b, "\nDoc3 tokens:", c)
print("WMD Doc1 vs Doc2:", vectors.wmdistance(a, b))
print("WMD Doc1 vs Doc3:", vectors.wmdistance(a, c))


---

## Program 15: Text Classification Tfidf

**Course outcome:** CO2


In [ ]:
# Program 15: Text classification using Naive Bayes/SVM with TF-IDF
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
texts = ["team won cricket match","player scored a goal","coach planned training","football player trained","cricket match starts","team lost final game","new phone camera","software update fixed laptop","computer has fast memory","laptop battery lasts long","phone has bright screen","install new software"]
labels = ["sports"]*6 + ["technology"]*6
a,b,y,z = train_test_split(texts, labels, test_size=.33, random_state=42, stratify=labels)
for name, clf in [("Naive Bayes", MultinomialNB()), ("Linear SVM", LinearSVC())]:
    model = Pipeline([("tfidf", TfidfVectorizer()), ("clf", clf)])
    model.fit(a, y); pred = model.predict(b)
    print("\n", name, "accuracy:", round(accuracy_score(z, pred), 3))
    print(classification_report(z, pred, zero_division=0))


---

## Program 16: Sentiment Textblob Vader

**Course outcome:** CO2


In [ ]:
# Program 16: Sentiment analysis using TextBlob and VADER
# Colab: !pip install -q textblob vaderSentiment
from textblob import TextBlob
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
reviews = ["This product is excellent and easy to use!", "The service was terrible and disappointing.", "The item arrived today and works as expected."]
vader = SentimentIntensityAnalyzer()
def label(x):
    return "Positive" if x > .05 else "Negative" if x < -.05 else "Neutral"
for r in reviews:
    tb = TextBlob(r).sentiment.polarity
    vd = vader.polarity_scores(r)["compound"]
    print("\nReview:", r)
    print(f"TextBlob: {tb:.3f} ({label(tb)})")
    print(f"VADER: {vd:.3f} ({label(vd)})")


---

## Program 17: Topic Modeling Lda

**Course outcome:** CO2


In [ ]:
# Program 17: Topic modeling using Latent Dirichlet Allocation (LDA)
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
docs = ["football team scored goals match", "coach trained football players", "python code uses machine learning", "developers build learning systems", "team won football game", "data science uses python machine learning"]
v = CountVectorizer(stop_words="english")
x = v.fit_transform(docs)
model = LatentDirichletAllocation(n_components=2, random_state=42, max_iter=20).fit(x)
terms = v.get_feature_names_out()
for i, weights in enumerate(model.components_, 1):
    print(f"Topic {i}:", ", ".join(terms[j] for j in weights.argsort()[-6:][::-1]))
print("Document-topic distributions:\n", model.transform(x).round(3))


---

## Program 18: Topic Modeling Lsa

**Course outcome:** CO2


In [ ]:
# Program 18: Topic modeling using Latent Semantic Analysis (LSA)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
docs = ["football players train match", "team scored goals football", "python machine learning data science", "learning models analyze data", "football team won match", "data science uses python"]
v = TfidfVectorizer(stop_words="english")
x = v.fit_transform(docs)
model = TruncatedSVD(n_components=2, random_state=42).fit(x)
terms = v.get_feature_names_out()
for i, weights in enumerate(model.components_, 1):
    print(f"Topic {i}:", ", ".join(terms[j] for j in weights.argsort()[-6:][::-1]))
print("Document-topic representation:\n", model.transform(x).round(3))


---

## Program 19: Opinion Mining Reviews

**Course outcome:** CO2


In [ ]:
# Program 19: Opinion mining on product/service reviews
import pandas as pd
from textblob import TextBlob
reviews = [
("Headphones", "Excellent sound quality and comfortable fit"),
("Headphones", "Battery is poor and connection drops"),
("Laptop", "Fast performance and beautiful display"),
("Laptop", "Expensive and gets very hot"),
("Delivery", "Delivery was quick and package was safe"),
("Delivery", "Parcel arrived late and support was unhelpful")]
rows = []
for product, review in reviews:
    p = TextBlob(review).sentiment.polarity
    sentiment = "Positive" if p > .05 else "Negative" if p < -.05 else "Neutral"
    rows.append({"product": product, "review": review, "polarity": round(p, 3), "sentiment": sentiment})
df = pd.DataFrame(rows)
print(df.to_string(index=False))
print("\nSentiment counts by product/service:\n", pd.crosstab(df.product, df.sentiment))


---

## Program 20: Information Extraction

**Course outcome:** CO2


In [ ]:
# Program 20: Information extraction from structured/unstructured documents
# Colab: !python -m spacy download en_core_web_sm
import re, spacy
nlp = spacy.load("en_core_web_sm")
text = """On 14 March 2026, NovaTech announced that it hired 25 engineers in New Delhi.
Contact hiring@novatech.example or call +91 98765 43210. The budget is Rs. 12,50,000."""
doc = nlp(text)
print("Named entities:")
for e in doc.ents: print(f"{e.text} -> {e.label_}")
print("Emails:", re.findall(r"\b[\w.+-]+@[\w.-]+\.[A-Za-z]{2,}\b", text))
print("Phone numbers:", re.findall(r"\+?\d[\d\s-]{8,}\d", text))
print("Budget:", re.findall(r"(?:Rs\.?|INR|₹)\s?[\d,]+", text))


---

## Program 21: Information Retrieval Tfidf

**Course outcome:** CO2


In [ ]:
# Program 21: Information retrieval with TF-IDF ranking
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
docs = ["Python is popular for data science and machine learning.", "NLP helps computers work with human text.", "Deep learning uses neural networks.", "Information retrieval ranks relevant documents.", "TF-IDF represents documents with weighted word frequencies."]
query = "document ranking using TF-IDF and information retrieval"
v = TfidfVectorizer(stop_words="english")
x = v.fit_transform(docs); q = v.transform([query])
scores = cosine_similarity(q, x).ravel()
for rank, i in enumerate(scores.argsort()[::-1], 1):
    print(f"{rank}. Score={scores[i]:.3f} | {docs[i]}")


---

## Program 22: Rnn Lstm Sequence Classification

**Course outcome:** CO2


In [ ]:
# Program 22: Sequence classification using RNN/LSTM (TensorFlow)
# Colab: !pip install -q tensorflow
import numpy as np, tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
texts = ["love this movie","wonderful film","great acting and story","excellent film","fantastic movie","hate this movie","terrible film","boring story","awful film","horrible movie"]
y = np.array([1]*5 + [0]*5, dtype="float32")
tok = Tokenizer(num_words=1000, oov_token="<OOV>"); tok.fit_on_texts(texts)
X = pad_sequences(tok.texts_to_sequences(texts), maxlen=8, padding="post")
tf.random.set_seed(42)
model = Sequential([Embedding(1000, 16), LSTM(16), Dense(1, activation="sigmoid")])
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=["accuracy"])
model.fit(X, y, epochs=12, verbose=0)
print("Training accuracy:", round(model.evaluate(X, y, verbose=0)[1], 3))
test = ["fantastic movie"]
seq = pad_sequences(tok.texts_to_sequences(test), maxlen=8, padding="post")
score = float(model.predict(seq, verbose=0)[0][0])
print(test[0], "->", "Positive" if score >= .5 else "Negative", "probability:", round(score, 3))


---

## Program 23: Attention Mechanism

**Course outcome:** CO2


In [ ]:
# Program 23: Implement a basic custom attention mechanism
import numpy as np
tokens = ["The", "student", "learns", "NLP"]
embeddings = np.array([[.2,.1,.7],[.8,.3,.2],[.4,.9,.1],[.6,.2,.8]])
query = np.array([.5,.7,.2])
scores = embeddings @ query
exp_scores = np.exp(scores - scores.max())
weights = exp_scores / exp_scores.sum()
context = weights @ embeddings
for token, score, weight in zip(tokens, scores, weights):
    print(f"{token:10} score={score:.3f} attention={weight:.3f}")
print("Context vector:", context.round(3))


---

## Program 24: Bert Text Classification

**Course outcome:** CO3


In [ ]:
# Program 24: Fine-tuning BERT for text classification using Hugging Face
# Colab: !pip install -q transformers datasets accelerate
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
model_name = "distilbert-base-uncased"
texts = ["I love this product","wonderful experience","service is excellent","very happy with result","fantastic quality","I hate this product","terrible experience","service is awful","disappointed with result","poor quality"]
labels = [1]*5 + [0]*5
ds = Dataset.from_dict({"text": texts, "label": labels}).train_test_split(test_size=.2, seed=42)
tok = AutoTokenizer.from_pretrained(model_name)
def encode(batch): return tok(batch["text"], truncation=True, padding="max_length", max_length=64)
ds = ds.map(encode, batched=True).remove_columns(["text"])
ds.set_format("torch")
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
args = TrainingArguments(output_dir="./bert-result", eval_strategy="epoch", save_strategy="no", num_train_epochs=1, per_device_train_batch_size=4, per_device_eval_batch_size=4, report_to="none")
trainer = Trainer(model=model, args=args, train_dataset=ds["train"], eval_dataset=ds["test"])
trainer.train()
print(trainer.evaluate())
text = "The product is wonderful and I am happy"
inputs = tok(text, return_tensors="pt", truncation=True, padding=True)
print(text, "->", ["Negative", "Positive"][int(model(**inputs).logits.argmax(-1))])


---

## Program 25: Sentence Embeddings Bert Gpt2

**Course outcome:** CO3


In [ ]:
# Program 25: Sentence embeddings using BERT and GPT-2
# Colab: !pip install -q transformers torch
import torch
from transformers import AutoTokenizer, AutoModel
sentences = ["NLP studies human language.", "NLP enables computers to process text."]
def pool(hidden, mask):
    mask = mask.unsqueeze(-1).expand(hidden.size()).float()
    return (hidden * mask).sum(1) / mask.sum(1).clamp(min=1e-9)
for name in ["bert-base-uncased", "gpt2"]:
    print("\nModel:", name)
    tok = AutoTokenizer.from_pretrained(name); model = AutoModel.from_pretrained(name)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    encoded = tok(sentences, return_tensors="pt", padding=True, truncation=True)
    with torch.no_grad(): out = model(**encoded)
    emb = torch.nn.functional.normalize(pool(out.last_hidden_state, encoded["attention_mask"]), p=2, dim=1)
    print("Embedding shape:", tuple(emb.shape))
    print("Cosine similarity:", round(float((emb[0] * emb[1]).sum()), 3))


---

## Program 26: Text Generation Gpt2

**Course outcome:** CO3


In [ ]:
# Program 26: Text generation using GPT-2
# Colab: !pip install -q transformers torch
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
name = "distilgpt2"
tok = AutoTokenizer.from_pretrained(name); model = AutoModelForCausalLM.from_pretrained(name)
prompt = "Natural language processing allows computers to"
inputs = tok(prompt, return_tensors="pt")
torch.manual_seed(42)
out = model.generate(**inputs, max_new_tokens=50, do_sample=True, temperature=.8, top_p=.9, pad_token_id=tok.eos_token_id)
print(tok.decode(out[0], skip_special_tokens=True))


---

## Program 27: Transformer Ner

**Course outcome:** CO3


In [ ]:
# Program 27: Named Entity Recognition using Transformers
# Colab: !pip install -q transformers torch
from transformers import AutoTokenizer, AutoModelForTokenClassification, pipeline
name = "dslim/bert-base-NER"
tok = AutoTokenizer.from_pretrained(name); model = AutoModelForTokenClassification.from_pretrained(name)
ner = pipeline("ner", model=model, tokenizer=tok, aggregation_strategy="simple", device=-1)
text = "Satya Nadella works at Microsoft in Redmond and visited London."
print("Text:", text)
for e in ner(text): print(f"{e['word']} -> {e['entity_group']} ({e['score']:.3f})")


---

## Program 28: Machine Translation Marianmt

**Course outcome:** CO3


In [ ]:
# Program 28: Machine translation using MarianMT
# Colab: !pip install -q transformers sentencepiece torch
from transformers import MarianMTModel, MarianTokenizer
name = "Helsinki-NLP/opus-mt-en-hi"
tok = MarianTokenizer.from_pretrained(name); model = MarianMTModel.from_pretrained(name)
sentences = ["Natural language processing is useful.", "Artificial intelligence can help students learn."]
inputs = tok(sentences, return_tensors="pt", padding=True, truncation=True)
out = model.generate(**inputs, max_new_tokens=80)
for src, pred in zip(sentences, out):
    print("English:", src)
    print("Hindi:", tok.decode(pred, skip_special_tokens=True), "\n")


---

## Program 29: Text Summarization Bart

**Course outcome:** CO3


In [ ]:
# Program 29: Text summarization using BART
# Colab: !pip install -q transformers torch
from transformers import pipeline
summarizer = pipeline("summarization", model="sshleifer/distilbart-cnn-12-6", device=-1)
text = """Natural language processing is a branch of artificial intelligence that enables computers to analyze, understand, and generate human language. Common tasks include tokenization, sentiment analysis, translation, summarization, information retrieval, and question answering. Modern systems often use neural networks and transformer architectures. Quality depends on data, model design, evaluation, and context."""
print(summarizer(text, max_length=55, min_length=20, do_sample=False)[0]["summary_text"])


---

## Program 30: Transformer Chatbot

**Course outcome:** CO3


In [ ]:
# Program 30: Chatbot using pretrained DialoGPT
# Colab: !pip install -q transformers torch
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
name = "microsoft/DialoGPT-small"
tok = AutoTokenizer.from_pretrained(name); model = AutoModelForCausalLM.from_pretrained(name)
history = None
print("Type 'quit' to stop.")
while True:
    text = input("You: ").strip()
    if text.lower() in {"quit", "exit"}:
        print("Bot: Goodbye!"); break
    new = tok.encode(text + tok.eos_token, return_tensors="pt")
    context = torch.cat([history, new], dim=-1) if history is not None else new
    history = model.generate(context, max_length=min(context.shape[-1] + 60, 900), pad_token_id=tok.eos_token_id, do_sample=True, top_p=.92, temperature=.75)
    print("Bot:", tok.decode(history[:, context.shape[-1]:][0], skip_special_tokens=True))


---

## Program 31: Document Classification Separation

**Course outcome:** CO3


In [ ]:
# Program 31: Automatic document classification and separation using ML
from pathlib import Path
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
texts = ["football match player team goal", "cricket batsman wicket tournament", "basketball player scored points", "python software database server", "computer hardware programming application", "machine learning algorithm data model", "doctor hospital patient treatment", "nurse clinic health medical care", "diagnosis medicine hospital doctor"]
labels = ["Sports"]*3 + ["Technology"]*3 + ["Healthcare"]*3
model = Pipeline([("tfidf", TfidfVectorizer()), ("svm", LinearSVC())]).fit(texts, labels)
docs = {"doc_01.txt":"The cricket team won the tournament.", "doc_02.txt":"The application uses Python and a database.", "doc_03.txt":"The doctor discussed treatment at the hospital."}
root = Path("classified_documents")
for filename, content in docs.items():
    category = model.predict([content])[0]
    folder = root / category; folder.mkdir(parents=True, exist_ok=True)
    (folder / filename).write_text(content, encoding="utf-8")
    print(f"{filename} -> {category}/{filename}")
print("Output folder:", root.resolve())


---

## Program 32: Hybrid Probabilistic Fsm Boundary Detection

**Course outcome:** Additional


In [ ]:
# Program 32: Hybrid probabilistic + FSM-based sequence modeling for document boundary detection
import re
text = """UNIVERSITY REPORT
Department of Computer Science
This report describes NLP laboratory work.
1. Introduction
Natural language processing analyzes text.
The system detects boundaries between sections.
CONCLUSION
The method combines heading cues with a finite-state machine."""
lines = [line.strip() for line in text.splitlines() if line.strip()]
def heading_probability(line):
    score = 0.0
    if len(line) < 70: score += .25
    if line.isupper(): score += .35
    if re.match(r"^\d+(\.\d+)*\.?\s+[A-Z]", line): score += .35
    if line.endswith(":"): score += .15
    if line.endswith(".") and not line.isupper(): score -= .15
    return max(0.0, min(1.0, score))
state, sections, current = "OUTSIDE_SECTION", [], None
for line in lines:
    probability = heading_probability(line)
    if probability >= .55:
        if current: sections.append(current)
        current = {"heading": line, "body": [], "probability": round(probability, 2)}
        state = "INSIDE_SECTION"
    else:
        if current is None:
            current = {"heading": "Preamble", "body": [], "probability": 0.0}
        current["body"].append(line)
        state = "INSIDE_SECTION"
if current: sections.append(current)
for i, section in enumerate(sections, 1):
    print(f"\n{i}. {section['heading']} (probability={section['probability']})")
    for line in section["body"]: print("  ", line)
